In [1]:
import time
t0 = time.time()

import sys
import zipfile
from pathlib import Path

import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import regionmask

# =========================
# 0. 当前服务器目录
# =========================
base = Path(".").resolve()

area_file = base / "grain_area_panel_2007_2023_cleaned.csv"
prod_file = base / "final_dataset_region_climate_with_coords.csv"
nc_file = base / "all_data_rain_temp.nc"
zip_file = base / "gadm41_RUS_shp.zip"

# 输出文件
out_csv = base / "final_panel_with_region_avg_climate.csv"
out_xlsx = base / "final_panel_with_region_avg_climate.xlsx"
out_monthly_csv = base / "climate_monthly_region_avg.csv"
out_yearly_csv = base / "climate_yearly_region_avg.csv"

# =========================
# 1. 解压 shp
# =========================
shp_dir = base / "gadm41_RUS_shp"
if not shp_dir.exists():
    shp_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(shp_dir)

shp_file = shp_dir / "gadm41_RUS_1.shp"

# =========================
# 2. 检查文件
# =========================
for f in [area_file, prod_file, nc_file, zip_file, shp_file]:
    if not f.exists():
        raise FileNotFoundError(f"没找到文件: {f}")

print("当前目录:", base)
print("面积表:", area_file)
print("产量表:", prod_file)
print("nc 文件:", nc_file)
print("shp 文件:", shp_file)

# =========================
# 3. 读取原始 csv
# =========================
area_df = pd.read_csv(area_file)
prod_df = pd.read_csv(prod_file)

area_df["region"] = area_df["region"].astype(str).str.strip()
area_df["year"] = pd.to_numeric(area_df["year"], errors="coerce").astype("Int64")

prod_df = prod_df[["region", "year", "production"]].copy()
prod_df["region"] = prod_df["region"].astype(str).str.strip()
prod_df["year"] = pd.to_numeric(prod_df["year"], errors="coerce").astype("Int64")

panel = area_df[["region", "year", "area_total"]].merge(
    prod_df,
    on=["region", "year"],
    how="left"
)

panel = panel[(panel["year"] >= 2007) & (panel["year"] <= 2023)].copy()
panel = panel.sort_values(["region", "year"]).reset_index(drop=True)
panel["region_std"] = panel["region"].astype(str).str.strip()

# =========================
# 4. 读取 Russia polygon
# =========================
gdf = gpd.read_file(shp_file)

if gdf.crs is None:
    raise ValueError("shp 没有 CRS。")

gdf = gdf.to_crs(epsg=4326)

if "NL_NAME_1" in gdf.columns:
    gdf["region_std"] = gdf["NL_NAME_1"].astype(str).str.strip()
elif "NAME_1" in gdf.columns:
    gdf["region_std"] = gdf["NAME_1"].astype(str).str.strip()
else:
    raise ValueError(f"shp 中没有名称列。列名为: {list(gdf.columns)}")

# 修正常见 GADM 俄文口径
shp_fix = {
    "Чукотский АОк": "Чукотский автономный округ",
    "Ханты-Мансийский АОк": "Ханты-Мансийский автономный округ",
    "Ямало-Ненецкий АОк": "Ямало-Ненецкий автономный округ",
    "Камчатская край": "Камчатский край",
    "Карачаево-Черкессия Республика": "Карачаево-Черкесская Республика",
    "Республика Саха": "Республика Саха (Якутия)",
    "Республика Северная Осетия-Алани": "Республика Северная Осетия-Алания",
    "Чувашская Республика": "Чувашская Республика - Чувашия",
    "Санкт-Петербург (горсовет)": "г. Санкт-Петербург",
    "Кемеровская область": "Кемеровская область - Кузбасс",
    "Пермская край": "Пермский край",
    "Республика Чечено-Ингушская": "Чеченская Республика",
    "Респу́блика Ингуше́тия": "Республика Ингушетия",
}
gdf["region_std"] = gdf["region_std"].replace(shp_fix)
gdf["region_std"] = gdf["region_std"].astype(str).str.strip()

# 自动丢掉 polygon 里没有的地区
panel_regions = set(panel["region_std"].dropna().unique())
shp_regions = set(gdf["region_std"].dropna().unique())

missing_in_shp = sorted(panel_regions - shp_regions)
if missing_in_shp:
    print("\n以下地区在 shp 中没有，将自动丢弃：")
    print(missing_in_shp)

panel = panel[panel["region_std"].isin(shp_regions)].copy()
gdf = gdf[gdf["region_std"].isin(panel["region_std"].unique())].copy()

print("\n保留地区数:", panel["region_std"].nunique())

# =========================
# 5. 读取 nc 文件
# =========================
ds = xr.open_dataset(nc_file)

# 自动识别坐标和时间
lon_name = None
lat_name = None
time_name = None

for cand in ["longitude", "lon", "x"]:
    if cand in ds.coords or cand in ds.dims:
        lon_name = cand
        break

for cand in ["latitude", "lat", "y"]:
    if cand in ds.coords or cand in ds.dims:
        lat_name = cand
        break

for cand in ["time", "valid_time"]:
    if cand in ds.coords or cand in ds.dims:
        time_name = cand
        break

if lon_name is None or lat_name is None or time_name is None:
    raise ValueError(f"无法识别坐标/时间维度。coords={list(ds.coords)}, dims={list(ds.dims)}")

# 经度转到 [-180,180)
if float(ds[lon_name].max()) > 180:
    ds = ds.assign_coords(
        {lon_name: (((ds[lon_name] + 180) % 360) - 180)}
    ).sortby(lon_name)

# 自动识别变量
temp_var = None
prec_var = None

for cand in ["t2m", "2m_temperature", "temp", "temperature"]:
    if cand in ds.data_vars:
        temp_var = cand
        break

for cand in ["tp", "total_precipitation", "precipitation", "rain"]:
    if cand in ds.data_vars:
        prec_var = cand
        break

if temp_var is None or prec_var is None:
    raise ValueError(f"无法识别温度/降水变量。当前变量：{list(ds.data_vars)}")

print("\n识别到变量:")
print("温度变量:", temp_var)
print("降水变量:", prec_var)

# 温度：K -> °C
ds["temp_c"] = ds[temp_var] - 273.15

# 降水：按月累计 mm
days_in_month = ds[time_name].dt.days_in_month
ds["prec_mm"] = ds[prec_var] * 1000.0 * days_in_month

# =========================
# 6. polygon 面平均月气候
# =========================
regions = regionmask.from_geopandas(gdf, names="region_std", name="russia_regions")
mask_3d = regions.mask_3D(ds[lon_name], ds[lat_name])

records = []

region_list = gdf["region_std"].tolist()
n_regions = len(region_list)

for i, region_name in enumerate(region_list, start=1):
    print(f"处理地区 {i}/{n_regions}: {region_name}")
    reg_mask = mask_3d.isel(region=i - 1)

    reg_temp = ds["temp_c"].where(reg_mask).mean(dim=(lat_name, lon_name), skipna=True)
    reg_prec = ds["prec_mm"].where(reg_mask).mean(dim=(lat_name, lon_name), skipna=True)

    temp_df = reg_temp.to_dataframe(name="temp_c").reset_index()
    prec_df = reg_prec.to_dataframe(name="prec_mm").reset_index()

    tmp = temp_df.merge(
        prec_df[[time_name, "prec_mm"]],
        on=time_name,
        how="inner"
    )
    tmp["region_std"] = region_name
    records.append(tmp[[time_name, "region_std", "temp_c", "prec_mm"]])

climate_monthly = pd.concat(records, ignore_index=True)
climate_monthly[time_name] = pd.to_datetime(climate_monthly[time_name])
climate_monthly["year"] = climate_monthly[time_name].dt.year
climate_monthly["month"] = climate_monthly[time_name].dt.month

climate_monthly = climate_monthly[["region_std", "year", "month", "temp_c", "prec_mm"]].copy()
climate_monthly = climate_monthly.sort_values(["region_std", "year", "month"]).reset_index(drop=True)
climate_monthly.to_csv(out_monthly_csv, index=False, encoding="utf-8-sig")

# =========================
# 7. 聚合成年气候变量
# =========================
cm = climate_monthly.copy()

annual = (
    cm.groupby(["region_std", "year"], as_index=False)
      .agg(
          temp_annual_mean=("temp_c", "mean"),
          prec_annual_sum=("prec_mm", "sum")
      )
)

grow = cm[cm["month"].between(4, 9)].copy()
grow = (
    grow.groupby(["region_std", "year"], as_index=False)
        .agg(
            temp_grow_mean=("temp_c", "mean"),
            prec_grow_sum=("prec_mm", "sum")
        )
)

cm["winter_year"] = cm["year"]
cm.loc[cm["month"].isin([11, 12]), "winter_year"] = cm["year"] + 1

winter = cm[cm["month"].isin([11, 12, 1, 2, 3])].copy()
winter = (
    winter.groupby(["region_std", "winter_year"], as_index=False)
          .agg(prec_winter_sum=("prec_mm", "sum"))
          .rename(columns={"winter_year": "year"})
)

climate_yearly = annual.merge(grow, on=["region_std", "year"], how="left")
climate_yearly = climate_yearly.merge(winter, on=["region_std", "year"], how="left")
climate_yearly = climate_yearly.sort_values(["region_std", "year"]).reset_index(drop=True)

climate_yearly["temp_annual_lag1"] = climate_yearly.groupby("region_std")["temp_annual_mean"].shift(1)
climate_yearly["prec_annual_lag1"] = climate_yearly.groupby("region_std")["prec_annual_sum"].shift(1)
climate_yearly["temp_grow_lag1"] = climate_yearly.groupby("region_std")["temp_grow_mean"].shift(1)
climate_yearly["prec_grow_lag1"] = climate_yearly.groupby("region_std")["prec_grow_sum"].shift(1)

climate_yearly.to_csv(out_yearly_csv, index=False, encoding="utf-8-sig")

# =========================
# 8. 合并回主面板
# =========================
panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)
climate_yearly["year"] = pd.to_numeric(climate_yearly["year"], errors="coerce").astype(int)

final_panel = panel.merge(
    climate_yearly,
    on=["region_std", "year"],
    how="left"
)

final_panel = final_panel[[
    "region", "region_std", "year", "production", "area_total",
    "temp_annual_mean", "prec_annual_sum",
    "temp_grow_mean", "prec_grow_sum",
    "prec_winter_sum",
    "temp_annual_lag1", "prec_annual_lag1",
    "temp_grow_lag1", "prec_grow_lag1"
]].copy()

final_panel = final_panel.sort_values(["region", "year"]).reset_index(drop=True)

# =========================
# 9. 输出
# =========================
final_panel.to_csv(out_csv, index=False, encoding="utf-8-sig")
final_panel.to_excel(out_xlsx, index=False)

print("\n完成。")
print("输出文件：")
print(out_csv)
print(out_xlsx)
print(out_monthly_csv)
print(out_yearly_csv)

print("\n最终表前10行：")
print(final_panel.head(10))

print("\n气候缺失统计：")
print(final_panel[[
    "temp_annual_mean", "prec_annual_sum",
    "temp_grow_mean", "prec_grow_sum", "prec_winter_sum"
]].isna().sum())

print("\n总耗时（分钟）:", round((time.time() - t0) / 60, 2))

当前目录: /root
面积表: /root/grain_area_panel_2007_2023_cleaned.csv
产量表: /root/final_dataset_region_climate_with_coords.csv
nc 文件: /root/all_data_rain_temp.nc
shp 文件: /root/gadm41_RUS_shp/gadm41_RUS_1.shp

以下地区在 shp 中没有，将自动丢弃：
['Еврейская автономная область', 'Кемеровская область', 'Республика Адыгея (Адыгея)', 'Республика Крым', 'Республика Татарстан (Татарстан)', 'Тюменская область (кроме Ханты-Мансийского автономного округа-Югры и Ямало-Ненецкого автономного округа)', 'Чувашская Республика', 'г. Москва', 'г. Севастополь']

保留地区数: 75

识别到变量:
温度变量: t2m
降水变量: tp
处理地区 1/75: Республика Адыгея
处理地区 2/75: Алтайский край
处理地区 3/75: Амурская область
处理地区 4/75: Архангельская область
处理地区 5/75: Астраханская область
处理地区 6/75: Республика Башкортостан
处理地区 7/75: Белгородская область
处理地区 8/75: Брянская область
处理地区 9/75: Республика Бурятия
处理地区 10/75: Чеченская Республика
处理地区 11/75: Челябинская область
处理地区 12/75: Чувашская Республика - Чувашия
处理地区 13/75: Республика Дагестан
处理地区 14/75: Республика А

In [2]:
df = pd.read_csv("/root/final_panel_with_region_avg_climate.csv")

print(df.sort_values("area_total", ascending=False).head(20)[["region", "year", "area_total", "production"]])
print(df[df["area_total"] <= 0][["region", "year", "area_total"]].head(20))

                    region  year    area_total  production
771        Республика Коми  2020  1.000000e+09         NaN
756     Республика Карелия  2018  1.000000e+09         NaN
48   Архангельская область  2021  1.000000e+09         NaN
772        Республика Коми  2021  1.000000e+09         NaN
770        Республика Коми  2019  1.000000e+09         NaN
769        Республика Коми  2018  1.000000e+09         NaN
2           Алтайский край  2009  3.803968e+03   5627845.0
1           Алтайский край  2008  3.781666e+03   3857490.0
10          Алтайский край  2017  3.746324e+03   4975500.0
7           Алтайский край  2014  3.717010e+03   3294900.0
905     Ростовская область  2022  3.673390e+03  15252000.0
906     Ростовская область  2023  3.655152e+03  16170300.0
9           Алтайский край  2016  3.646225e+03   4829700.0
903     Ростовская область  2020  3.639472e+03  12464500.0
8           Алтайский край  2015  3.632101e+03   3940400.0
4           Алтайский край  2011  3.628322e+03   3919500

In [3]:
import pandas as pd
import numpy as np

# =========================
# 1. 读取文件
# =========================
monthly_file = "/root/climate_monthly_region_avg.csv"
panel_file = "/root/final_panel_with_region_avg_climate.csv"

monthly = pd.read_csv(monthly_file)
panel = pd.read_csv(panel_file)

print("monthly shape:", monthly.shape)
print("panel shape:", panel.shape)

# =========================
# 2. 修正降水单位
# 当前 monthly 里的 prec_mm 实际上偏小
# 这里按“月平均 -> 月累计”修正：
# 月累计(mm) = 原值 * 当月天数
# =========================
monthly["year"] = pd.to_numeric(monthly["year"], errors="coerce").astype(int)
monthly["month"] = pd.to_numeric(monthly["month"], errors="coerce").astype(int)

# 给每一行补当月天数
monthly["date"] = pd.to_datetime(
    monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2) + "-01"
)
monthly["days_in_month"] = monthly["date"].dt.days_in_month

# 修正后的月降水
monthly["prec_mm_fixed"] = monthly["prec_mm"] * monthly["days_in_month"]

# =========================
# 3. 重新聚合成年气候变量
# =========================

# 全年
annual = (
    monthly.groupby(["region_std", "year"], as_index=False)
    .agg(
        temp_annual_mean=("temp_c", "mean"),
        prec_annual_sum_fixed=("prec_mm_fixed", "sum")
    )
)

# 生长季 4-9月
grow = monthly[monthly["month"].between(4, 9)].copy()
grow = (
    grow.groupby(["region_std", "year"], as_index=False)
    .agg(
        temp_grow_mean=("temp_c", "mean"),
        prec_grow_sum_fixed=("prec_mm_fixed", "sum")
    )
)

# 冬季水分：上一年11-12月 + 当年1-3月
winter = monthly.copy()
winter["winter_year"] = winter["year"]
winter.loc[winter["month"].isin([11, 12]), "winter_year"] = winter["year"] + 1
winter = winter[winter["month"].isin([11, 12, 1, 2, 3])].copy()

winter = (
    winter.groupby(["region_std", "winter_year"], as_index=False)
    .agg(
        prec_winter_sum_fixed=("prec_mm_fixed", "sum")
    )
    .rename(columns={"winter_year": "year"})
)

# 合并
climate_fixed = annual.merge(grow, on=["region_std", "year"], how="left")
climate_fixed = climate_fixed.merge(winter, on=["region_std", "year"], how="left")
climate_fixed = climate_fixed.sort_values(["region_std", "year"]).reset_index(drop=True)

# lag 变量
climate_fixed["temp_annual_lag1"] = climate_fixed.groupby("region_std")["temp_annual_mean"].shift(1)
climate_fixed["prec_annual_lag1_fixed"] = climate_fixed.groupby("region_std")["prec_annual_sum_fixed"].shift(1)
climate_fixed["temp_grow_lag1"] = climate_fixed.groupby("region_std")["temp_grow_mean"].shift(1)
climate_fixed["prec_grow_lag1_fixed"] = climate_fixed.groupby("region_std")["prec_grow_sum_fixed"].shift(1)

print("climate_fixed shape:", climate_fixed.shape)

# =========================
# 4. 删掉旧的错误降水列，换成修正后的
# =========================
panel_fixed = panel.drop(
    columns=[
        "prec_annual_sum",
        "prec_grow_sum",
        "prec_winter_sum",
        "prec_annual_lag1",
        "prec_grow_lag1",
    ],
    errors="ignore"
).copy()

panel_fixed = panel_fixed.drop(
    columns=[
        "temp_annual_mean",
        "temp_grow_mean",
        "temp_annual_lag1",
        "temp_grow_lag1",
    ],
    errors="ignore"
).copy()

panel_fixed = panel_fixed.merge(
    climate_fixed,
    on=["region_std", "year"],
    how="left"
)

# =========================
# 5. 清洗面积和产量异常值
# =========================

# 删掉面积哨兵值 1e9
panel_fixed = panel_fixed[panel_fixed["area_total"] < 1e8].copy()

# 删掉面积 <= 0
panel_fixed = panel_fixed[panel_fixed["area_total"] > 0].copy()

# 删掉产量缺失或 <= 0
panel_fixed = panel_fixed[panel_fixed["production"].notna()].copy()
panel_fixed = panel_fixed[panel_fixed["production"] > 0].copy()

# 删掉关键气候变量缺失
panel_fixed = panel_fixed.dropna(subset=[
    "temp_annual_mean",
    "prec_annual_sum_fixed",
    "temp_grow_mean",
    "prec_grow_sum_fixed",
    "prec_winter_sum_fixed"
]).copy()

# =========================
# 6. 构造回归变量
# =========================
panel_fixed["ln_production"] = np.log(panel_fixed["production"])
panel_fixed["ln_area"] = np.log(panel_fixed["area_total"])

# 可选：二次项
panel_fixed["temp_grow_sq"] = panel_fixed["temp_grow_mean"] ** 2
panel_fixed["prec_grow_sq"] = panel_fixed["prec_grow_sum_fixed"] ** 2

# =========================
# 7. 重命名列，方便后续回归
# =========================
panel_fixed = panel_fixed.rename(columns={
    "prec_annual_sum_fixed": "prec_annual_sum",
    "prec_grow_sum_fixed": "prec_grow_sum",
    "prec_winter_sum_fixed": "prec_winter_sum",
    "prec_annual_lag1_fixed": "prec_annual_lag1",
    "prec_grow_lag1_fixed": "prec_grow_lag1",
})

# =========================
# 8. 排序并输出
# =========================
panel_fixed = panel_fixed.sort_values(["region", "year"]).reset_index(drop=True)

out_csv = "/root/final_panel_regression_ready.csv"
out_xlsx = "/root/final_panel_regression_ready.xlsx"

panel_fixed.to_csv(out_csv, index=False, encoding="utf-8-sig")
panel_fixed.to_excel(out_xlsx, index=False)

print("\n清洗完成")
print("输出文件：")
print(out_csv)
print(out_xlsx)

print("\n最终 shape:", panel_fixed.shape)

print("\n缺失统计：")
print(panel_fixed[[
    "production", "area_total",
    "temp_annual_mean", "prec_annual_sum",
    "temp_grow_mean", "prec_grow_sum",
    "prec_winter_sum",
    "temp_annual_lag1", "prec_annual_lag1",
    "temp_grow_lag1", "prec_grow_lag1"
]].isna().sum())

print("\n描述统计：")
print(panel_fixed[[
    "production", "area_total",
    "temp_annual_mean", "prec_annual_sum",
    "temp_grow_mean", "prec_grow_sum",
    "prec_winter_sum"
]].describe())

monthly shape: (15300, 5)
panel shape: (1220, 14)
climate_fixed shape: (1275, 11)

清洗完成
输出文件：
/root/final_panel_regression_ready.csv
/root/final_panel_regression_ready.xlsx

最终 shape: (1170, 18)

缺失统计：
production           0
area_total           0
temp_annual_mean     0
prec_annual_sum      0
temp_grow_mean       0
prec_grow_sum        0
prec_winter_sum      0
temp_annual_lag1    71
prec_annual_lag1    71
temp_grow_lag1      71
prec_grow_lag1      71
dtype: int64

描述统计：
         production   area_total  temp_annual_mean  prec_annual_sum  \
count  1.170000e+03  1170.000000       1170.000000      1170.000000   
mean   1.546249e+06   651.629975          4.453790     19834.663330   
std    2.383622e+06   839.298473          4.665868      6349.139250   
min    1.000000e+01     0.008000        -10.617039      5422.393316   
25%    1.415000e+05    83.013177          2.340540     15747.831990   
50%    5.527600e+05   242.526000          5.477958     19409.721825   
75%    2.082275e+06   940.32

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("/root/final_panel_regression_ready.csv")

# 基准 FE
m1 = smf.ols(
    "ln_production ~ temp_grow_mean + prec_grow_sum + C(region) + C(year)",
    data=df
).fit()

print(m1.summary())

# 二次项 FE
m2 = smf.ols(
    "ln_production ~ temp_grow_mean + temp_grow_sq + prec_grow_sum + prec_grow_sq + C(region) + C(year)",
    data=df
).fit()

print(m2.summary())

                            OLS Regression Results                            
Dep. Variable:          ln_production   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.978
Method:                 Least Squares   F-statistic:                     578.9
Date:                Mon, 30 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:16:44   Log-Likelihood:                -307.46
No. Observations:                1170   AIC:                             792.9
Df Residuals:                    1081   BIC:                             1244.
Df Model:                          88                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [2]:
import pandas as pd

df = pd.read_csv("/root/final_panel_regression_ready.csv")

# =========================
# 1. 农业生态区分组
# 可按老师要求先粗分为：
# - black_soil: 黑钙土区
# - volga_dry: 干旱伏尔加区/南乌拉尔干旱农业区
# - risky_farming: 高风险耕作区（北部、寒冷、湿冷、林区）
# - other: 其他
# =========================
zone_map = {
    # 黑钙土区
    "Белгородская область": "black_soil",
    "Воронежская область": "black_soil",
    "Курская область": "black_soil",
    "Липецкая область": "black_soil",
    "Тамбовская область": "black_soil",
    "Орловская область": "black_soil",
    "Ростовская область": "black_soil",
    "Краснодарский край": "black_soil",
    "Ставропольский край": "black_soil",
    "Республика Татарстан": "black_soil",
    "Республика Татарстан (Татарстан)": "black_soil",
    "Республика Башкортостан": "black_soil",
    "Пензенская область": "black_soil",
    "Самарская область": "black_soil",
    "Саратовская область": "black_soil",
    "Ульяновская область": "black_soil",
    "Оренбургская область": "black_soil",
    "Алтайский край": "black_soil",

    # 干旱伏尔加区 / 南部干旱农业区
    "Астраханская область": "volga_dry",
    "Волгоградская область": "volga_dry",
    "Саратовская область": "volga_dry",
    "Самарская область": "volga_dry",
    "Оренбургская область": "volga_dry",
    "Республика Калмыкия": "volga_dry",
    "Ставропольский край": "volga_dry",

    # 高风险耕作区
    "Архангельская область": "risky_farming",
    "Вологодская область": "risky_farming",
    "Мурманская область": "risky_farming",
    "Ленинградская область": "risky_farming",
    "Республика Карелия": "risky_farming",
    "Республика Коми": "risky_farming",
    "Пермский край": "risky_farming",
    "Кировская область": "risky_farming",
    "Новгородская область": "risky_farming",
    "Псковская область": "risky_farming",
    "Томская область": "risky_farming",
    "Иркутская область": "risky_farming",
    "Республика Саха (Якутия)": "risky_farming",
    "Камчатский край": "risky_farming",
    "Хабаровский край": "risky_farming",
    "Магаданская область": "risky_farming",
    "Сахалинская область": "risky_farming",
    "Чукотский автономный округ": "risky_farming",
    "Ямало-Ненецкий автономный округ": "risky_farming",
    "Ханты-Мансийский автономный округ": "risky_farming",
}

df["agro_zone"] = df["region"].map(zone_map).fillna("other")

print(df[["region", "agro_zone"]].drop_duplicates().sort_values(["agro_zone", "region"]).head(100))
print(df["agro_zone"].value_counts())

df.to_csv("/root/final_panel_regression_ready_with_zone.csv", index=False, encoding="utf-8-sig")
df.to_excel("/root/final_panel_regression_ready_with_zone.xlsx", index=False)

                   region   agro_zone
0          Алтайский край  black_soil
59   Белгородская область  black_soil
144   Воронежская область  black_soil
328    Краснодарский край  black_soil
379       Курская область  black_soil
..                    ...         ...
515  Оренбургская область   volga_dry
717   Республика Калмыкия   volga_dry
898     Самарская область   volga_dry
915   Саратовская область   volga_dry
966   Ставропольский край   volga_dry

[71 rows x 2 columns]
agro_zone
other            625
black_soil       219
risky_farming    209
volga_dry        117
Name: count, dtype: int64


In [3]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("/root/final_panel_regression_ready_with_zone.csv")

# 为了稳一点，把 agro_zone 设成分类变量
df["agro_zone"] = df["agro_zone"].astype("category")
df["region"] = df["region"].astype("category")
df["year"] = df["year"].astype("category")

# =========================
# 模型1：基准 FE
# =========================
m1 = smf.ols(
    "ln_production ~ temp_grow_mean + prec_grow_sum + C(region) + C(year)",
    data=df
).fit()

print("\n================ 模型1：基准 FE ================\n")
print(m1.summary())

# =========================
# 模型2：二次项 FE
# =========================
m2 = smf.ols(
    "ln_production ~ temp_grow_mean + temp_grow_sq + prec_grow_sum + prec_grow_sq + C(region) + C(year)",
    data=df
).fit()

print("\n================ 模型2：二次项 FE ================\n")
print(m2.summary())

# =========================
# 模型3：农业生态区主效应 + 气候交互
# 注意：有地区固定效应时，agro_zone 主效应本身会被吸收，
# 但交互项仍然有意义
# =========================
m3 = smf.ols(
    "ln_production ~ temp_grow_mean * C(agro_zone) + prec_grow_sum * C(agro_zone) + C(region) + C(year)",
    data=df
).fit()

print("\n================ 模型3：气候 × 农业生态区 ================\n")
print(m3.summary())

# =========================
# 模型4：二次项 + 农业生态区交互
# =========================
m4 = smf.ols(
    "ln_production ~ temp_grow_mean * C(agro_zone) + prec_grow_sum * C(agro_zone) + temp_grow_sq + prec_grow_sq + C(region) + C(year)",
    data=df
).fit()

print("\n================ 模型4：二次项 + 气候 × 农业生态区 ================\n")
print(m4.summary())


================ 模型1：基准 FE ================

                            OLS Regression Results                            
Dep. Variable:          ln_production   R-squared:                       0.979
Model:                            OLS   Adj. R-squared:                  0.978
Method:                 Least Squares   F-statistic:                     578.9
Date:                Mon, 30 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:17:44   Log-Likelihood:                -307.46
No. Observations:                1170   AIC:                             792.9
Df Residuals:                    1081   BIC:                             1244.
Df Model:                          88                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------

In [4]:
def tidy_result(model, model_name):
    out = pd.DataFrame({
        "coef": model.params,
        "t": model.tvalues,
        "p": model.pvalues
    })
    out["model"] = model_name
    return out

res = pd.concat([
    tidy_result(m1, "m1_base_fe"),
    tidy_result(m2, "m2_quad_fe"),
    tidy_result(m3, "m3_zone_interaction"),
    tidy_result(m4, "m4_quad_zone_interaction"),
])

# 只看核心变量
keep = [
    "temp_grow_mean",
    "prec_grow_sum",
    "temp_grow_sq",
    "prec_grow_sq",
]
res_core = res[res.index.str.contains("temp_grow_mean|prec_grow_sum|temp_grow_sq|prec_grow_sq|agro_zone", regex=True)].copy()

print(res_core)
res_core.to_csv("/root/regression_results_core.csv", encoding="utf-8-sig")

                                                      coef          t  \
temp_grow_mean                               -1.147902e-01  -6.752395   
prec_grow_sum                                 7.729050e-06   1.287001   
temp_grow_mean                               -2.886431e-02  -0.533394   
temp_grow_sq                                 -2.687466e-03  -1.393459   
prec_grow_sum                                 5.839654e-05   4.100505   
prec_grow_sq                                 -1.693378e-09  -3.987993   
C(agro_zone)[T.other]                        -3.706475e+00  -6.927650   
C(agro_zone)[T.risky_farming]                -6.553369e+00 -11.269265   
C(agro_zone)[T.volga_dry]                    -7.935655e-01  -0.940919   
temp_grow_mean                               -1.807918e-01  -5.689225   
temp_grow_mean:C(agro_zone)[T.other]          8.973332e-02   2.646600   
temp_grow_mean:C(agro_zone)[T.risky_farming]  9.188193e-02   2.190579   
temp_grow_mean:C(agro_zone)[T.volga_dry]      2.942

In [5]:
import sys
!{sys.executable} -m pip install -U statsmodels pandas numpy scipy patsy

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 18.4 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 11.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.24.1
    Uninstalling numpy-1.24.1:
      Successfully uninstalled numpy-1.24.1


In [1]:
from pathlib import Path

cdsapirc = """url: https://cds.climate.copernicus.eu/api
key: 8f52c490-8058-4847-8c92-112ae3dfc83d
"""

Path("/root/.cdsapirc").write_text(cdsapirc)
print("已写入 /root/.cdsapirc")

已写入 /root/.cdsapirc


In [2]:
print(Path("/root/.cdsapirc").read_text())

url: https://cds.climate.copernicus.eu/api
key: 8f52c490-8058-4847-8c92-112ae3dfc83d



In [6]:
import os, certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

print("certifi CA bundle =", certifi.where())

certifi CA bundle = /root/miniconda3/lib/python3.10/site-packages/certifi/cacert.pem


In [7]:
import cdsapi
import certifi
from pathlib import Path

out_file = Path("/root/all_data_rain_temp.nc")

client = cdsapi.Client(
    url="https://cds.climate.copernicus.eu/api",
    verify=certifi.where()
)

request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [
        "2m_temperature",
        "total_precipitation",
    ],
    "year": [str(y) for y in range(2007, 2024)],
    "month": [f"{m:02d}" for m in range(1, 13)],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [82, 18, 41, 191],
}

client.retrieve(
    "reanalysis-era5-land-monthly-means",
    request,
    str(out_file)
)

print("下载完成：", out_file)

2026-03-30 05:41:24,078 INFO Request ID is 7418aa84-e145-4258-a686-93684cf11711
2026-03-30 05:41:24,270 INFO status has been updated to accepted
2026-03-30 05:41:57,762 INFO status has been updated to running
2026-03-30 05:42:15,038 INFO status has been updated to successful


f4f2dee2f4d5466d888eb1d5f804e2bd.nc:   0%|          | 0.00/320M [00:00<?, ?B/s]

下载完成： /root/all_data_rain_temp.nc


In [8]:
import xarray as xr
from pathlib import Path

nc_file = Path("/root/all_data_rain_temp.nc")
print("存在吗:", nc_file.exists())
print("大小(MB):", round(nc_file.stat().st_size / 1024**2, 2))

ds = xr.open_dataset(nc_file, engine="netcdf4")
print(ds)
print(ds.data_vars)

存在吗: True
大小(MB): 319.89
<xarray.Dataset> Size: 1GB
Dimensions:     (valid_time: 204, latitude: 411, longitude: 1731)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 2kB 2007-01-01 ... 2023-12-01
  * latitude    (latitude) float64 3kB 82.0 81.9 81.8 81.7 ... 41.2 41.1 41.0
  * longitude   (longitude) float64 14kB 18.0 18.1 18.2 ... 190.8 190.9 191.0
    expver      (valid_time) <U4 3kB ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 581MB ...
    tp          (valid_time, latitude, longitude) float32 581MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-29T21:36 GRIB to CDM+CF via cfgrib-0.9.1...
Data variables:
    t2m      (valid_time, latitude, longitude) 